## CNN using embeddings to predict log binding affinity - testing at different protein similarity thresholds

### Load the data:

In [1]:
import pandas as pd
import numpy as np
import random
import json
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold, ParameterGrid
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, r2_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel, EsmTokenizer, EsmModel
from tqdm import tqdm

2025-02-10 16:33:30.133016: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-02-10 16:33:34.408359: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /local/java/cuda-11.6.0/lib64/:/local/java/cudnn-linux-x86_64-8.5.0.96_cuda11-archive/lib/:/local/java/cuda-12.6.2/lib64/:/local/java/cudnn-linux-x86_64-9.5.1.17_cuda12-archive/lib/
2025-02-10 16:33:34.409373: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared objec

In [2]:
# Preprocess the data
data = pd.read_csv('/dcs/22/u2243582/cs310/feature_extraction/refined-set-csv.csv')
data = data[~data['Protein_FASTA'].str.contains('X')] # remove complexes with ambiguous char in FASTA string
toDrop = ['4yx4', '1laf', '4buq', '1k22', '1hmt', '1utn', '4rux', '1bty']
data = data[~data['PDB_Code'].isin(toDrop)]
# data = data.reset_index()
display(data)

,PDB_Code,Protein_FASTA,Ligand_SMILES,Binding_data,Log_binding
0,6ugp,HWGYGKHNGPEHWHKDFPIAKGERQSPVDIDTHTAKYDPSLKPLSV...,O=C1Nc2c(Cl)cccc2S(=O)(=O)N1,Ki=685.5nM,6.16
1,4rdn,ENLYFQHMKHGRVFIIKSYSEDDIHRSIKYNIWCSTEHGNKRLDAA...,CNc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O,Kd=1.2uM,5.92
2,4mo4,PRYLGLMSGTSLDGMDIVLIEQGDRTTLLASHYLPMPAGLREDILA...,C[P@@](=O)([O-])O[P@@](=O)([O-])OC[C@H]1O[C@@H...,Kd=0.06mM,4.22
3,3s0b,MTIEELKTRLHTEQSVCKTETGIDQQKANDVIEGNIDVEDKKVQLY...,c1ccc(Nc2cccc3ccccc23)cc1,Kd=0.32uM,6.49
4,6r1d,ASIFRCRQCGQTISRRDWLLPMGGDHEHVVFNPAGMIFRVWCFSLA...,O=C1C[C@H](NC(=O)OCc2ccccc2)C(=O)N1,Ki=4uM,5.40
...,...,...,...,...,...
5244,6p3t,VTLCSPTEDDWPGMFLLAAASFTDFIGPESATAWRTLVPTDGAVVV...,CN(c1ccc2ccccc2c1)S(=O)(=O)c1ccc2[nH]c(=O)c(=O...,Ki=0.048uM,7.32
5245,1tx7,IVGGYTCGANTVPYQVSLNSGYHFCGGSLINSQWVVSAAHCYKSGI...,C[P@@](=O)([O-])c1ccc(C(N)=[NH2+])cc1,Ki=25uM,4.60
5246,3ta1,MKMVVAVIRPEKLECVKKALEERGFVGMTVTEVKGRGEQKGIRLQF...,Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO[P@@](=O)([O-])O...,Kd=568uM,3.25
5247,6r0v,VLAPGASIFRCRQCGQTISRRDWLLPMGGDHEHVVFNPAGMIFRVW...,O=C1C[C@H](NC(=O)c2cc([N+](=O)[O-])ccc2C(=O)[O...,Ki=11uM,4.96


In [3]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
# To generate embeddings, need list of FASTA and list of SMILES
protein_fastas = data['Protein_FASTA'].tolist()
ligand_SMILES = data['Ligand_SMILES'].tolist()

# Load protein embedding model (ESM2)
esm_model_name = "facebook/esm2_t12_35M_UR50D"
esm_tokenizer = EsmTokenizer.from_pretrained(esm_model_name)
esm_model = EsmModel.from_pretrained(esm_model_name).to(device)

# Load ligand embedding model (ChemBERTa)
chemberta_model_name = "DeepChem/ChemBERTa-10M-MTR"
chemberta_tokenizer = AutoTokenizer.from_pretrained(chemberta_model_name)
chemberta_model = AutoModel.from_pretrained(chemberta_model_name).to(device)

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t12_35M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-10M-MTR and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
# Batch processing for protein sequences
def generate_protein_embeddings(fasta_sequences, batch_size=4, max_length=1024):
    esm_model.eval()
    all_embeddings = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(fasta_sequences), batch_size), desc="Processing Protein Batches"):
            batch = fasta_sequences[i:i + batch_size]
            inputs = esm_tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
            outputs = esm_model(**inputs)
            # Extract the last layer and apply mean pooling
            embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            all_embeddings.append(embeddings)
            # Clear GPU memory after processing the batch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    return np.concatenate(all_embeddings, axis=0)

# Batch processing for ligand SMILES strings
def generate_ligand_embeddings(smiles_list, batch_size=4):
    chemberta_model.eval()
    all_embeddings = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(smiles_list), batch_size), desc="Processing Ligand Batches"):
            batch = smiles_list[i:i + batch_size]
            inputs = chemberta_tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)
            outputs = chemberta_model(**inputs)
            # Extract the last layer and apply mean pooling
            embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            all_embeddings.append(embeddings)
            # Clear GPU memory after processing the batch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    return np.concatenate(all_embeddings, axis=0)

In [6]:
# Generate Embeddings
protein_embeddings = generate_protein_embeddings(protein_fastas)
print(f"\nProtein embeddings shape: {protein_embeddings.shape}")

ligand_embeddings = generate_ligand_embeddings(ligand_SMILES)
print(f"Ligand embeddings shape: {ligand_embeddings.shape}")

# Single array of embeddings data
embeddings = np.concatenate((protein_embeddings, ligand_embeddings), axis=1) # (5059, 864)
print(embeddings.shape)

Processing Protein Batches: 100%|██████████| 1265/1265 [01:26<00:00, 14.66it/s]



Protein embeddings shape: (5059, 480)


Processing Ligand Batches: 100%|██████████| 1265/1265 [00:04<00:00, 313.57it/s]

Ligand embeddings shape: (5059, 384)
(5059, 864)


In [7]:
# Scale the data
scaler = StandardScaler()
scaler.fit(embeddings)
X_scaled = scaler.transform(embeddings)

In [8]:
# Final dataframe of concatted embeddings, PDB code identifier and log binding affinity value
X_scaled = pd.DataFrame(X_scaled, index=data.index)
data_final = X_scaled.join(data['PDB_Code'])
data_final = data_final.join(data['Log_binding'])
display(data_final)

,0,1,2,3,4,5,6,7,8,9,...,856,857,858,859,860,861,862,863,PDB_Code,Log_binding
0,-0.553410,1.102691,2.021120,1.411761,1.214011,0.227725,-0.595541,0.227407,-0.748729,-0.100622,...,-0.727279,-0.354272,-1.496568,-1.677355,-1.263457,-1.499772,0.018008,-0.079486,6ugp,6.16
1,-0.967858,-1.834613,0.740057,0.149697,1.607742,-0.312229,-1.037183,0.508405,-0.564584,0.369930,...,-0.295743,0.732035,-0.104872,0.290937,0.983046,-0.412831,0.960099,0.788265,4rdn,5.92
2,1.350682,-1.125857,-2.232252,-1.327422,-2.076709,1.762392,2.814136,-0.114218,-0.751788,-1.638773,...,-0.570634,0.028024,-0.852041,1.900848,0.769626,0.463825,1.911561,0.163847,4mo4,4.22
3,-0.306406,0.264238,0.055367,-0.631101,1.425525,-0.273559,-1.051705,0.755581,-0.647675,0.706085,...,-1.134623,0.420818,-0.284302,-0.279638,0.474691,0.797058,-0.785853,-0.705691,3s0b,6.49
4,0.627834,0.116611,-0.884056,-0.410392,0.493873,0.662111,-0.561305,-0.468997,1.433913,0.702334,...,-1.262924,0.388194,-0.572381,-0.754786,-0.770169,1.770026,-0.011041,-1.828241,6r1d,5.40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5244,1.575801,-1.510840,-1.288181,0.967735,-0.960101,-0.780322,0.263016,-1.810867,0.104862,1.584121,...,0.843475,1.433339,0.469883,-0.377851,0.850565,0.878081,0.055095,0.581299,6p3t,7.32
5245,-1.707911,-0.068936,0.600451,0.274773,1.716015,0.579620,-0.931325,0.854444,-1.484255,-1.795299,...,-1.839617,-2.239669,0.517412,0.506642,0.993042,-0.040314,-0.791730,0.352069,1tx7,4.60
5246,-0.296805,-2.620062,1.902834,-0.377663,-0.047489,-0.843551,-0.223779,-0.787270,2.312208,0.385790,...,-0.721253,0.970989,-1.443581,2.234030,1.636725,1.052548,3.096036,0.985693,3ta1,3.25
5247,0.227410,1.447401,-0.518395,-0.458092,0.256863,0.554503,0.370857,-0.469208,0.924484,1.061119,...,0.291097,-0.068378,-0.108143,-0.204578,-1.723246,0.319448,-0.488850,0.563393,6r0v,4.96


In [9]:
import random
import numpy as np
def NRKFold(E,pc,K = 5, shuffle=True):
    """
    Generate non-redundant K-folds for a dataset where each example involves an object 
    that belongs to a certain cluster. This function ensures that no two folds contain 
    objects from the same cluster and aims to distribute the number of examples 
    approximately equally across all folds.

    This is particularly useful in scenarios where data points can naturally group into 
    clusters (e.g., proteins in bioinformatics), and it is important to avoid having 
    similar examples in both the training and validation sets of a particular fold.

    Parameters
    ----------
    E : List
        A list containing identifiers of objects involved in each example.
    pc : Dictionary
        A dictionary mapping each object to its cluster assignment.
    K : Integer, optional
        The number of folds to create. Default is 5.
    shuffle : Boolean, optional
        Determines whether to shuffle the cluster to fold assignments in different runs.
        Default is True.

    Returns
    -------
    List of lists
        A list where each sublist contains the indices of examples in `objects` that belong to a particular fold.

    Example
    -------
    >>> objects = ['obj1', 'obj2', 'obj3', 'obj4', 'obj5', 'obj6', 'obj1']
    >>> clusters = {'obj1': 1, 'obj2': 2, 'obj3': 1, 'obj4': 2, 'obj5': 3, 'obj6': 3}
    >>> folds = NRKFold(objects, clusters, K=2, shuffle=False)
    >>> print(folds)
    Output might be: [[0, 2, 6], [1, 3, 4, 5]]
    Here, objects 'obj1', 'obj3', and 'obj1' (indices 0, 2, 6) are in one fold, 
    and the rest are in another fold, ensuring no fold has objects from the same cluster.
    """
    e = [pc[str(x)] for x in E] #cluster indices of all proteins in the examples
    c2idx={} #indices of examples of each cluster in e
    for i,x in enumerate(e):
        try: 
            c2idx[x].append(i)
        except:
            c2idx[x]=[i]    
    ce = dict([(c,len(c2idx[c])) for c in c2idx]) #counts of examples of different clusters    
    cF = [0]*K; #counts of examples in each fold
    CF = [[] for _ in range(K)]; #clusters in each fold
    F = [[] for _ in range(K)];#indices of examples in each fold
    keys = list(ce.keys())
    if shuffle:
        random.shuffle(keys)
    for k in keys:
        v = ce[k]
        idx = np.argmin(cF)
        cF[idx]+=v
        CF[idx].append(k) #add cluster to fold
        F[idx].extend(c2idx[k])
    return F

In [53]:
with open("/dcs/22/u2243582/cs310/nrkf_physicochem_feat/clustered_proteins.json", "r") as json_file:
    cluster_dict = json.load(json_file)

print(len(cluster_dict)) # length of dict should match the output of num of clusters from clustering program

cluster_assignment_dict = {}
for cluster_ind in cluster_dict:
    for pdb in cluster_dict[cluster_ind]:
        cluster_assignment_dict[pdb] = cluster_ind
        
identifiers = list(cluster_assignment_dict.keys())

1192


In [54]:
folds = NRKFold(identifiers, cluster_assignment_dict)

In [55]:
final_folds = []
for fold in folds:
    temp_fold = []
    for obj in fold:
        pdb = identifiers[obj]
        data_ind = data.index[data['PDB_Code'] == pdb].item()
        temp_fold.append(data_ind)
    final_folds.append(temp_fold)

In [56]:
# The final NRKfolds:
splits = [
    {
        "train_ix": final_folds[0] + final_folds[1] + final_folds[2] + final_folds[3],
        "test_ix": final_folds[4]
    },
    {
        "train_ix": final_folds[0] + final_folds[1] + final_folds[2] + final_folds[4],
        "test_ix": final_folds[3]
    },
    {
       "train_ix": final_folds[0] + final_folds[1] + final_folds[4] + final_folds[3],
        "test_ix": final_folds[2]
    },
    {
        "train_ix": final_folds[0] + final_folds[4] + final_folds[2] + final_folds[3],
        "test_ix": final_folds[1]
    },
    {
        "train_ix": final_folds[4] + final_folds[1] + final_folds[2] + final_folds[3],
        "test_ix": final_folds[0]
    }
]

### CNN:

In [15]:
class CNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_filters, kernel_size, dropout_rate):
        super(CNNModel, self).__init__()

        self.hidden_dim = hidden_dim  

        self.conv1 = nn.Conv1d(in_channels=1, out_channels=num_filters, kernel_size=kernel_size, stride=1, padding=kernel_size//2)
        self.bn1 = nn.BatchNorm1d(num_filters)

        self.conv2 = nn.Conv1d(in_channels=num_filters, out_channels=num_filters * 2, kernel_size=kernel_size, stride=1, padding=kernel_size//2)
        self.bn2 = nn.BatchNorm1d(num_filters * 2)

        self.conv3 = nn.Conv1d(in_channels=num_filters * 2, out_channels=num_filters * 4, kernel_size=kernel_size, stride=1, padding=kernel_size//2)
        self.bn3 = nn.BatchNorm1d(num_filters * 4)

        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)  # Dropout layer

        self.flatten_dim = self._get_flatten_dim(input_dim, num_filters, kernel_size)
        
        # Fully connected layers
        self.fc1 = nn.Linear(self.flatten_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)  # New fully connected layer
        self.fc3 = nn.Linear(hidden_dim // 2, 1)

    def _get_flatten_dim(self, input_dim, num_filters, kernel_size):
        # Helper function to compute the output size dynamically
        
        sample_input = torch.randn(1, 1, input_dim)  
        
        x = self.pool(self.relu(self.bn1(self.conv1(sample_input))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))

        return x.view(1, -1).size(1)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)

        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)

        x = self.relu(self.bn3(self.conv3(x)))
        x = self.pool(x)

        x = x.view(x.size(0), -1)  # Flatten tensor to [batch_size, flattened_dim]

        x = self.relu(self.fc1(x))
        x = self.dropout(x) 

        x = self.relu(self.fc2(x))
        # x = self.dropout(x) 

        x = self.fc3(x)

        return x.squeeze()

In [16]:
def train(model, train_loader, val_loader, criterion, optimizer, epochs=50, patience=10):
    
    model.to(device)
    best_loss = float("inf")
    patience_counter = 0
    epoch_losses = []
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        avg_train_loss = train_loss / len(train_loader)
        epoch_losses.append(avg_train_loss)

        # ---- Validation Step ----
        model.eval()
        val_loss = 0
        
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                y_pred = model(X_batch)
                val_loss += criterion(y_pred, y_batch).item()

        avg_val_loss = val_loss / len(val_loader)
        
        # ---- Early Stopping Check ----
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping after {epoch+1} epochs")
                break  

    return epoch_losses

In [20]:
def cross_validate(data, params, n_splits=5, batch_size=32, epochs=50):
    
    all_loss_curves = []
    count = 1

    results = []  # Store results

    for fold in splits:
        print(f"Fold {count}")
        
        train_ix = np.array(fold["train_ix"])
        test_ix = np.array(fold["test_ix"])
        
        X_train = data.loc[train_ix,  0:863].values
        X_test = data.loc[test_ix,  0:863].values
        y_train = data.loc[train_ix, 'Log_binding'].values
        y_test = data.loc[test_ix, 'Log_binding'].values #.reshape(-1,1)
        
        X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
        X_test = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)
        y_train = torch.tensor(y_train, dtype=torch.float32)
        y_test = torch.tensor(y_test, dtype=torch.float32)

        # Train/Validation Split 
        train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

        # Initialize model
        model = CNNModel(input_dim=864, 
                hidden_dim=params["hidden_dim"],
                num_filters=params["num_filters"],
                kernel_size=params["kernel_size"],
                dropout_rate=params["dropout_rate"]).to(device)
        criterion = nn.L1Loss()
        optimizer = optim.Adam(model.parameters(), lr=params["learning_rate"], weight_decay=0)

        # Train model
        epoch_losses = train(model, train_loader, test_loader, criterion, optimizer, epochs=epochs)
        all_loss_curves.append(epoch_losses)

        # ---- Evaluate Model ----
        model.eval()
        y_pred_list, y_true_list = [], []

        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.to(device)
                y_pred = model(X_batch).squeeze().cpu().numpy()
                y_true = y_batch.squeeze().cpu().numpy()

                y_pred_list.extend(y_pred)
                y_true_list.extend(y_true)

        pearson_corr, pearson_p = pearsonr(y_true_list, y_pred_list)
        spearman_corr, spearman_p = spearmanr(y_true_list, y_pred_list)
        mae = mean_absolute_error(y_true_list, y_pred_list)
        variance = np.var(np.array(y_true_list) - np.array(y_pred_list))
        r2 = r2_score(y_true_list, y_pred_list)

        results.append({"Fold": count, "Pearson": pearson_corr, "Pearson p": pearson_p, "Spearman": spearman_corr, "Spearman p": spearman_p, "MAE": mae, "Variance": variance, "R2": r2})
        count +=1
        
    return results

In [57]:
# Use modal hyperparameters from baseline nested cv:
params = {'hidden_dim': 256, 'dropout_rate': 0.2, 'learning_rate': 0.005, 'num_filters': 64, 'kernel_size': 3}

results = cross_validate(data_final, params, n_splits=5, batch_size=64, epochs=50)

Fold 1
Early stopping after 27 epochs
Fold 2
Early stopping after 16 epochs
Fold 3
Early stopping after 25 epochs
Fold 4
Early stopping after 18 epochs
Fold 5
Early stopping after 18 epochs


In [58]:
# Nested CV Extract metrics
pearson_scores = [result['Pearson'] for result in results]
spearman_scores = [result['Spearman'] for result in results]
mae_scores = [result['MAE'] for result in results]
var_scores = [result['Variance'] for result in results]
r2_scores = [result['R2'] for result in results]
pearson_p_values = [result['Pearson p'] for result in results]
spearman_p_values = [result['Spearman p'] for result in results]

# Calculate means and standard deviations
pearson_mean = np.mean(pearson_scores)
pearson_std = np.std(pearson_scores)

spearman_mean = np.mean(spearman_scores)
spearman_std = np.std(spearman_scores)

mae_mean = np.mean(mae_scores)
mae_std = np.std(mae_scores)

var_mean = np.mean(var_scores)
var_std = np.std(var_scores)

r2_mean = np.mean(r2_scores)
r2_std = np.std(r2_scores)

pearson_p_mean = np.mean(pearson_p_values)
pearson_p_std = np.std(pearson_p_values)

spearman_p_mean = np.mean(spearman_p_values)
spearman_p_std = np.std(spearman_p_values)

# Print the results
print(f"Pearson Correlation: Mean = {pearson_mean}, Std = {pearson_std}, p-value Mean = {pearson_p_mean}, p-value Std = {pearson_p_std}")
print(f"Spearman Correlation: Mean = {spearman_mean}, Std = {spearman_std}, p-value Mean = {spearman_p_mean}, p-value Std = {spearman_p_std}")
print(f"MAE: Mean = {mae_mean}, Std = {mae_std}")
print(f"Variance: Mean = {var_mean}, Std = {var_std}")
print(f"R2: Mean = {r2_mean}, Std = {r2_std}")

Pearson Correlation: Mean = 0.48624796457776237, Std = 0.045609122200716994, p-value Mean = 1.2983048699395515e-47, p-value Std = 2.544940608876343e-47
Spearman Correlation: Mean = 0.4850620091505621, Std = 0.05873502147900479, p-value Mean = 1.5329998991844776e-40, p-value Std = 3.0659997966064292e-40
MAE: Mean = 1.606292963027954, Std = 0.31600385904312134
Variance: Mean = 2.8345892429351807, Std = 0.6462823152542114
R2: Mean = -0.09761450195372907, Std = 0.24016939898276116
